# CD2 — Feature Engineering

## Proyecto EnergiAI — Inteligencia para el Consumo Energético

Este notebook documenta el proceso de ingeniería de características desarrollado por CD2 para preparar los datasets **Smart Home Energy Consumption** y **UCI Household Power Consumption** para la etapa de modelado CD3.

### Objetivos

- Preparar los datos provenientes de las fuentes originales.
- Construir variables temporales.
- Representar variables temporales cíclicas mediante seno y coseno.
- Codificar variables categóricas requeridas para modelado.
- Construir indicadores asociados con horarios de mayor consumo.
- Seleccionar las variables predictoras y las variables objetivo.
- Generar particiones de entrenamiento y prueba.
- Validar la integridad de los datasets resultantes.
- Exportar los artefactos finales destinados a CD3.

### Variables objetivo

**Smart Home**

`Energy Consumption (kWh)`

**UCI Household Power Consumption**

`Global_active_power`

### Estrategia de partición

- **Smart Home:** partición aleatoria reproducible 80/20 con `random_state=42`.
- **UCI:** partición cronológica 80/20 para preservar la naturaleza temporal de las observaciones.

In [1]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("\nEntorno preparado correctamente.")

Pandas: 2.2.2
NumPy: 2.0.2

Entorno preparado correctamente.


In [2]:
# ============================================================
# 2. CONFIGURACIÓN PORTABLE DE RUTAS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Localizar automáticamente la raíz del repositorio
# ------------------------------------------------------------

posibles_raices = [
    Path("/content/team30_repo"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent
]

REPO_ROOT = None

for ruta in posibles_raices:
    if (
        (ruta / "analisis_datos").exists()
        and (ruta / ".git").exists()
    ):
        REPO_ROOT = ruta.resolve()
        break

if REPO_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio.\n"
        "Clona el repositorio o ejecuta este notebook desde "
        "una ubicación perteneciente al proyecto."
    )

# ------------------------------------------------------------
# Directorios del proyecto
# ------------------------------------------------------------

ANALISIS_DIR = REPO_ROOT / "analisis_datos"

RAW_DIR = ANALISIS_DIR / "data" / "raw"

CD2_DIR = ANALISIS_DIR / "notebooks" / "cd2"
OUTPUT_DIR = CD2_DIR / "data"

REPORTS_DIR = ANALISIS_DIR / "reports" / "cd2"

# ------------------------------------------------------------
# Fuentes RAW
# ------------------------------------------------------------

SMART_RAW = (
    RAW_DIR /
    "smart_home_energy_consumption_large.csv"
)

UCI_RAW = (
    RAW_DIR /
    "household_power_consumption.csv"
)

# ------------------------------------------------------------
# Crear directorios de salida si no existen
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Verificación
# ------------------------------------------------------------

print("========== CONFIGURACIÓN DEL PROYECTO ==========\n")

print("Repositorio:")
print(REPO_ROOT)

print("\nDatos RAW:")
print(RAW_DIR)

print("\nSalida CD2:")
print(OUTPUT_DIR)

print("\nReportes CD2:")
print(REPORTS_DIR)

print("\n========== FUENTES ==========")

print(
    "Smart Home:",
    SMART_RAW,
    "\nExiste:",
    SMART_RAW.exists()
)

print(
    "\nUCI:",
    UCI_RAW,
    "\nExiste:",
    UCI_RAW.exists()
)

# Fallar explícitamente si falta alguna fuente
assert SMART_RAW.exists(), (
    f"No se encontró Smart Home RAW: {SMART_RAW}"
)

assert UCI_RAW.exists(), (
    f"No se encontró UCI RAW: {UCI_RAW}"
)

print("\nCONFIGURACIÓN PORTABLE VALIDADA CORRECTAMENTE")

========== CONFIGURACIÓN DEL PROYECTO ==========

Repositorio:
/content/team30_repo

Datos RAW:
/content/team30_repo/analisis_datos/data/raw

Salida CD2:
/content/team30_repo/analisis_datos/notebooks/cd2/data

Reportes CD2:
/content/team30_repo/analisis_datos/reports/cd2

========== FUENTES ==========
Smart Home: /content/team30_repo/analisis_datos/data/raw/smart_home_energy_consumption_large.csv 
Existe: True

UCI: /content/team30_repo/analisis_datos/data/raw/household_power_consumption.csv 
Existe: True

CONFIGURACIÓN PORTABLE VALIDADA CORRECTAMENTE


In [3]:
# ============================================================
# 3. SMART HOME — CARGA
# ============================================================

df_smart = pd.read_csv(SMART_RAW)

print("========== SMART HOME ==========")
print("Dimensiones RAW:", df_smart.shape)
print("Nulos:", df_smart.isna().sum().sum())
print("Duplicados:", df_smart.duplicated().sum())

display(df_smart.head())

========== SMART HOME ==========
Dimensiones RAW: (100000, 8)
Nulos: 0
Duplicados: 0


,Home ID,Appliance Type,Energy Consumption (kWh),Time,Date,Outdoor Temperature (°C),Season,Household Size
0,94,Fridge,0.20,21:12,2023-12-02,-1.0,Fall,2
1,435,Oven,0.23,20:11,2023-08-06,31.1,Summer,5
2,466,Dishwasher,0.32,06:39,2023-11-21,21.3,Fall,3
3,496,Heater,3.92,21:56,2023-01-21,-4.2,Winter,1
4,137,Microwave,0.44,04:31,2023-08-26,34.5,Summer,5


In [4]:
# ============================================================
# 4. SMART HOME — VARIABLES TEMPORALES
# ============================================================

df_smart["datetime"] = pd.to_datetime(
    df_smart["Date"].astype(str) + " " +
    df_smart["Time"].astype(str),
    errors="coerce"
)

df_smart["hour"] = df_smart["datetime"].dt.hour
df_smart["month"] = df_smart["datetime"].dt.month
df_smart["day_of_week"] = df_smart["datetime"].dt.dayofweek

df_smart["is_weekend"] = (
    df_smart["day_of_week"] >= 5
).astype(int)

print("Fechas no convertidas:", df_smart["datetime"].isna().sum())

print("\nDistribución is_weekend:")
print(df_smart["is_weekend"].value_counts().sort_index())

Fechas no convertidas: 0

Distribución is_weekend:
is_weekend
0    71095
1    28905
Name: count, dtype: int64


In [5]:
# ============================================================
# 5. SMART HOME — CODIFICACIÓN CÍCLICA
# ============================================================

df_smart["hour_sin"] = np.sin(
    2 * np.pi * df_smart["hour"] / 24
)

df_smart["hour_cos"] = np.cos(
    2 * np.pi * df_smart["hour"] / 24
)

df_smart["dow_sin"] = np.sin(
    2 * np.pi * df_smart["day_of_week"] / 7
)

df_smart["dow_cos"] = np.cos(
    2 * np.pi * df_smart["day_of_week"] / 7
)

df_smart["month_sin"] = np.sin(
    2 * np.pi * (df_smart["month"] - 1) / 12
)

df_smart["month_cos"] = np.cos(
    2 * np.pi * (df_smart["month"] - 1) / 12
)

variables_ciclicas_smart = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

print("Nulos variables cíclicas:")
print(df_smart[variables_ciclicas_smart].isna().sum())

print("\nRangos:")
display(
    df_smart[variables_ciclicas_smart]
    .agg(["min", "max"])
    .T
    .round(6)
)

Nulos variables cíclicas:
hour_sin     0
hour_cos     0
dow_sin      0
dow_cos      0
month_sin    0
month_cos    0
dtype: int64

Rangos:


,min,max
hour_sin,-1.000000,1.000000
hour_cos,-1.000000,1.000000
dow_sin,-0.974928,0.974928
dow_cos,-0.900969,1.000000
month_sin,-1.000000,1.000000
month_cos,-1.000000,1.000000


In [6]:
# ============================================================
# 6. SMART HOME — ONE-HOT ENCODING APPLIANCE TYPE
# ============================================================

appliance_encoded = pd.get_dummies(
    df_smart["Appliance Type"],
    prefix="appliance",
    dtype=int
)

df_smart = pd.concat(
    [df_smart, appliance_encoded],
    axis=1
)

print("Categorías Appliance Type:")
print(sorted(df_smart["Appliance Type"].unique()))

print("\nVariables One-Hot:")
print(appliance_encoded.columns.tolist())

print("\nCantidad:")
print(appliance_encoded.shape[1])

print("\nFilas inválidas One-Hot:")
print((appliance_encoded.sum(axis=1) != 1).sum())

Categorías Appliance Type:
['Air Conditioning', 'Computer', 'Dishwasher', 'Fridge', 'Heater', 'Lights', 'Microwave', 'Oven', 'TV', 'Washing Machine']

Variables One-Hot:
['appliance_Air Conditioning', 'appliance_Computer', 'appliance_Dishwasher', 'appliance_Fridge', 'appliance_Heater', 'appliance_Lights', 'appliance_Microwave', 'appliance_Oven', 'appliance_TV', 'appliance_Washing Machine']

Cantidad:
10

Filas inválidas One-Hot:
0


In [7]:
# ============================================================
# 7. SMART HOME — INDICADOR HORARIO PICO
# ============================================================

df_smart["is_peak_hour"] = (
    df_smart["hour"].isin([19, 20, 21, 22])
).astype(int)

print("Distribución is_peak_hour - Smart Home:")
print(df_smart["is_peak_hour"].value_counts().sort_index())

print("\nPorcentaje:")
print(
    (
        df_smart["is_peak_hour"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

Distribución is_peak_hour - Smart Home:
is_peak_hour
0    83460
1    16540
Name: count, dtype: int64

Porcentaje:
is_peak_hour
0    83.46
1    16.54
Name: proportion, dtype: float64


In [8]:
# ============================================================
# 8. SMART HOME — SELECCIÓN DE VARIABLES
# ============================================================

TARGET_SMART = "Energy Consumption (kWh)"

FEATURES_SMART = [
    "Outdoor Temperature (°C)",
    "Household Size",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "appliance_Air Conditioning",
    "appliance_Computer",
    "appliance_Dishwasher",
    "appliance_Fridge",
    "appliance_Heater",
    "appliance_Lights",
    "appliance_Microwave",
    "appliance_Oven",
    "appliance_TV",
    "appliance_Washing Machine",
    "is_peak_hour"
]

X_smart = df_smart[FEATURES_SMART].copy()
y_smart = df_smart[TARGET_SMART].copy()

print("========== SMART HOME FINAL ==========")
print("X:", X_smart.shape)
print("y:", y_smart.shape)

print("\nNúmero de predictoras:")
print(len(FEATURES_SMART))

print("\nNulos X:", X_smart.isna().sum().sum())
print("Nulos y:", y_smart.isna().sum())

print("\nPredictoras:")
for variable in FEATURES_SMART:
    print("-", variable)

========== SMART HOME FINAL ==========
X: (100000, 20)
y: (100000,)

Número de predictoras:
20

Nulos X: 0
Nulos y: 0

Predictoras:
- Outdoor Temperature (°C)
- Household Size
- is_weekend
- hour_sin
- hour_cos
- dow_sin
- dow_cos
- month_sin
- month_cos
- appliance_Air Conditioning
- appliance_Computer
- appliance_Dishwasher
- appliance_Fridge
- appliance_Heater
- appliance_Lights
- appliance_Microwave
- appliance_Oven
- appliance_TV
- appliance_Washing Machine
- is_peak_hour


In [9]:
# ============================================================
# 9. SMART HOME — TRAIN / TEST
# ============================================================

X_train_smart, X_test_smart, y_train_smart, y_test_smart = (
    train_test_split(
        X_smart,
        y_smart,
        test_size=0.20,
        random_state=42
    )
)

smart_train = X_train_smart.copy()
smart_train[TARGET_SMART] = y_train_smart

smart_test = X_test_smart.copy()
smart_test[TARGET_SMART] = y_test_smart

print("========== PARTICIÓN SMART HOME ==========")

print("TRAIN:", smart_train.shape)
print("TEST :", smart_test.shape)

print("\nNulos TRAIN:", smart_train.isna().sum().sum())
print("Nulos TEST:", smart_test.isna().sum().sum())

========== PARTICIÓN SMART HOME ==========
TRAIN: (80000, 21)
TEST : (20000, 21)

Nulos TRAIN: 0
Nulos TEST: 0


UCI HOUSEHOLD POWER CONSUMPTION

In [10]:
# ============================================================
# 10. UCI — CARGA
# ============================================================

df_uci = pd.read_csv(
    UCI_RAW,
    sep=";",
    low_memory=False
)

print("========== UCI RAW ==========")
print("Dimensiones RAW:", df_uci.shape)

print("\nColumnas:")
print(df_uci.columns.tolist())

display(df_uci.head())

========== UCI RAW ==========
Dimensiones RAW: (1048575, 1)

Columnas:
['Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3']


,"Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3"
0,"16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0,..."
1,"16/12/2006,17:25:00,5.36,0.436,233.63,23,0,1,16"
2,"16/12/2006,17:26:00,5.374,0.498,233.29,23,0,2,17"
3,"16/12/2006,17:27:00,5.388,0.502,233.74,23,0,1,17"
4,"16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0,..."


In [11]:
# ============================================================
# 11. UCI — LIMPIEZA DE VARIABLES ELÉCTRICAS
# ============================================================

df_uci = pd.read_csv(
    UCI_RAW,
    sep=",", # Corrected separator to comma
    low_memory=False
)

columnas_electricas_uci = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for columna in columnas_electricas_uci:
    df_uci[columna] = pd.to_numeric(
        df_uci[columna],
        errors="coerce"
    )

registros_antes = len(df_uci)

df_uci = df_uci.dropna(
    subset=columnas_electricas_uci
).copy()

registros_despues = len(df_uci)

print("Registros antes :", registros_antes)
print("Registros después:", registros_despues)
print("Registros eliminados:", registros_antes - registros_despues)

print(
    "Porcentaje eliminado:",
    round(
        (registros_antes - registros_despues)
        / registros_antes * 100,
        4
    ),
    "%"
)

print("\nNulos eléctricos:")
print(df_uci[columnas_electricas_uci].isna().sum())

Registros antes : 1048575
Registros después: 1044506
Registros eliminados: 4069
Porcentaje eliminado: 0.3881 %

Nulos eléctricos:
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64


In [12]:
# ============================================================
# 12. UCI — DATETIME Y ORDEN CRONOLÓGICO
# ============================================================

df_uci["datetime"] = pd.to_datetime(
    df_uci["Date"].astype(str) + " " +
    df_uci["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

print("Fechas no convertidas:")
print(df_uci["datetime"].isna().sum())

df_uci = (
    df_uci
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("\nFecha inicial:")
print(df_uci["datetime"].min())

print("\nFecha final:")
print(df_uci["datetime"].max())

print("\nOrden cronológico:")
print(df_uci["datetime"].is_monotonic_increasing)

print("\nTimestamps duplicados:")
print(df_uci["datetime"].duplicated().sum())

Fechas no convertidas:
0

Fecha inicial:
2006-12-16 17:24:00

Fecha final:
2008-12-13 21:38:00

Orden cronológico:
True

Timestamps duplicados:
0


In [13]:
# ============================================================
# 13. UCI — VARIABLES TEMPORALES
# ============================================================

df_uci["hour"] = df_uci["datetime"].dt.hour
df_uci["month"] = df_uci["datetime"].dt.month
df_uci["day_of_week"] = df_uci["datetime"].dt.dayofweek

df_uci["is_weekend"] = (
    df_uci["day_of_week"] >= 5
).astype(int)

df_uci["hour_sin"] = np.sin(
    2 * np.pi * df_uci["hour"] / 24
)

df_uci["hour_cos"] = np.cos(
    2 * np.pi * df_uci["hour"] / 24
)

df_uci["dow_sin"] = np.sin(
    2 * np.pi * df_uci["day_of_week"] / 7
)

df_uci["dow_cos"] = np.cos(
    2 * np.pi * df_uci["day_of_week"] / 7
)

df_uci["month_sin"] = np.sin(
    2 * np.pi * (df_uci["month"] - 1) / 12
)

df_uci["month_cos"] = np.cos(
    2 * np.pi * (df_uci["month"] - 1) / 12
)

print("Variables temporales creadas correctamente.")

Variables temporales creadas correctamente.


In [14]:
# ============================================================
# 14. UCI — INDICADOR HORARIO PICO
# ============================================================

df_uci["is_peak_hour"] = (
    df_uci["hour"].isin([19, 20, 21, 22])
).astype(int)

print("Distribución is_peak_hour - UCI:")
print(df_uci["is_peak_hour"].value_counts().sort_index())

print("\nPorcentaje:")
print(
    (
        df_uci["is_peak_hour"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

Distribución is_peak_hour - UCI:
is_peak_hour
0    870130
1    174376
Name: count, dtype: int64

Porcentaje:
is_peak_hour
0    83.31
1    16.69
Name: proportion, dtype: float64


In [15]:
# ============================================================
# 15. UCI — VARIABLES ENERGÉTICAS AUXILIARES
# ============================================================

# Energía global aproximada por minuto en Wh
df_uci["Energia_Global_Wh"] = (
    df_uci["Global_active_power"] * 1000 / 60
)

# Suma de los tres submedidores
df_uci["Suma_Submetering_Wh"] = (
    df_uci["Sub_metering_1"]
    + df_uci["Sub_metering_2"]
    + df_uci["Sub_metering_3"]
)

# Energía no explicada por los tres submedidores
df_uci["Energia_No_Submedida_Wh"] = (
    df_uci["Energia_Global_Wh"]
    - df_uci["Suma_Submetering_Wh"]
)

# Conversión utilizada para análisis agregado de energía
df_uci["energy_kwh"] = (
    df_uci["Global_active_power"] / 60
)

print("Variables energéticas auxiliares creadas.")

print("\nNulos:")
print(
    df_uci[
        [
            "Energia_Global_Wh",
            "Suma_Submetering_Wh",
            "Energia_No_Submedida_Wh",
            "energy_kwh"
        ]
    ].isna().sum()
)

Variables energéticas auxiliares creadas.

Nulos:
Energia_Global_Wh          0
Suma_Submetering_Wh        0
Energia_No_Submedida_Wh    0
energy_kwh                 0
dtype: int64


In [16]:
# ============================================================
# 16. UCI — SELECCIÓN DE VARIABLES
# ============================================================

TARGET_UCI = "Global_active_power"

FEATURES_UCI = [
    "Global_reactive_power",
    "Voltage",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "is_peak_hour"
]

X_uci = df_uci[FEATURES_UCI].copy()
y_uci = df_uci[TARGET_UCI].copy()

print("========== UCI FINAL ==========")

print("X:", X_uci.shape)
print("y:", y_uci.shape)

print("\nNúmero de predictoras:")
print(len(FEATURES_UCI))

print("\nNulos X:", X_uci.isna().sum().sum())
print("Nulos y:", y_uci.isna().sum())

print("\nPredictoras:")
for variable in FEATURES_UCI:
    print("-", variable)

========== UCI FINAL ==========
X: (1044506, 13)
y: (1044506,)

Número de predictoras:
13

Nulos X: 0
Nulos y: 0

Predictoras:
- Global_reactive_power
- Voltage
- Sub_metering_1
- Sub_metering_2
- Sub_metering_3
- is_weekend
- hour_sin
- hour_cos
- dow_sin
- dow_cos
- month_sin
- month_cos
- is_peak_hour


In [17]:
# ============================================================
# 17. UCI — PARTICIÓN TEMPORAL 80/20
# ============================================================

punto_corte = int(len(df_uci) * 0.80)

X_train_uci = X_uci.iloc[:punto_corte].copy()
X_test_uci = X_uci.iloc[punto_corte:].copy()

y_train_uci = y_uci.iloc[:punto_corte].copy()
y_test_uci = y_uci.iloc[punto_corte:].copy()

uci_train = X_train_uci.copy()
uci_train[TARGET_UCI] = y_train_uci

uci_test = X_test_uci.copy()
uci_test[TARGET_UCI] = y_test_uci

print("========== PARTICIÓN UCI ==========")

print("Punto de corte:", punto_corte)

print("TRAIN:", uci_train.shape)
print("TEST :", uci_test.shape)

print("\nFin TRAIN:")
print(df_uci.iloc[punto_corte - 1]["datetime"])

print("\nInicio TEST:")
print(df_uci.iloc[punto_corte]["datetime"])

print("\n¿TRAIN termina antes que TEST?:")
print(
    df_uci.iloc[punto_corte - 1]["datetime"]
    <
    df_uci.iloc[punto_corte]["datetime"]
)

========== PARTICIÓN UCI ==========
Punto de corte: 835604
TRAIN: (835604, 14)
TEST : (208902, 14)

Fin TRAIN:
2008-07-21 17:52:00

Inicio TEST:
2008-07-21 17:53:00

¿TRAIN termina antes que TEST?:
True


In [18]:
# ============================================================
# 18. VALIDACIÓN FINAL CD2
# ============================================================

resumen = pd.DataFrame(
    {
        "Dataset": [
            "Smart Home - Train",
            "Smart Home - Test",
            "UCI - Train",
            "UCI - Test"
        ],
        "Registros": [
            len(smart_train),
            len(smart_test),
            len(uci_train),
            len(uci_test)
        ],
        "Predictoras": [
            len(FEATURES_SMART),
            len(FEATURES_SMART),
            len(FEATURES_UCI),
            len(FEATURES_UCI)
        ],
        "Nulos": [
            smart_train.isna().sum().sum(),
            smart_test.isna().sum().sum(),
            uci_train.isna().sum().sum(),
            uci_test.isna().sum().sum()
        ]
    }
)

display(resumen)

print("\n========== OBJETIVOS ==========")
print("Smart Home:", TARGET_SMART)
print("UCI:", TARGET_UCI)

print("\n========== PARTICIÓN ==========")
print("Smart Home: aleatoria reproducible 80/20 - random_state=42")
print("UCI: cronológica 80/20")

,Dataset,Registros,Predictoras,Nulos
0,Smart Home - Train,80000,20,0
1,Smart Home - Test,20000,20,0
2,UCI - Train,835604,13,0
3,UCI - Test,208902,13,0



========== OBJETIVOS ==========
Smart Home: Energy Consumption (kWh)
UCI: Global_active_power

========== PARTICIÓN ==========
Smart Home: aleatoria reproducible 80/20 - random_state=42
UCI: cronológica 80/20


In [19]:
# ============================================================
# 19. PRUEBAS DE INTEGRIDAD
# ============================================================

assert smart_train.shape == (80000, 21)
assert smart_test.shape == (20000, 21)

assert uci_train.shape == (835604, 14)
assert uci_test.shape == (208902, 14)

assert smart_train.isna().sum().sum() == 0
assert smart_test.isna().sum().sum() == 0
assert uci_train.isna().sum().sum() == 0
assert uci_test.isna().sum().sum() == 0

assert TARGET_SMART in smart_train.columns
assert TARGET_SMART in smart_test.columns

assert TARGET_UCI in uci_train.columns
assert TARGET_UCI in uci_test.columns

assert len(FEATURES_SMART) == 20
assert len(FEATURES_UCI) == 13

assert (
    df_uci.iloc[punto_corte - 1]["datetime"]
    <
    df_uci.iloc[punto_corte]["datetime"]
)

print("============================================")
print(" TODAS LAS PRUEBAS DE INTEGRIDAD SUPERADAS")
print("============================================")

 TODAS LAS PRUEBAS DE INTEGRIDAD SUPERADAS


In [20]:
# ============================================================
# 20. EXPORTACIÓN DE VALIDACIÓN
# ============================================================

smart_train_path = OUTPUT_DIR / "smart_home_train_reproducido.csv"
smart_test_path = OUTPUT_DIR / "smart_home_test_reproducido.csv"

uci_train_path = OUTPUT_DIR / "uci_train_reproducido.csv"
uci_test_path = OUTPUT_DIR / "uci_test_reproducido.csv"

smart_train.to_csv(
    smart_train_path,
    index=False
)

smart_test.to_csv(
    smart_test_path,
    index=False
)

uci_train.to_csv(
    uci_train_path,
    index=False
)

uci_test.to_csv(
    uci_test_path,
    index=False
)

print("Archivos reproducidos correctamente:\n")

for archivo in [
    smart_train_path,
    smart_test_path,
    uci_train_path,
    uci_test_path
]:
    print(
        archivo.name,
        "-",
        round(archivo.stat().st_size / 1024**2, 2),
        "MB"
    )

Archivos reproducidos correctamente:

smart_home_train_reproducido.csv - 10.71 MB
smart_home_test_reproducido.csv - 2.68 MB
uci_train_reproducido.csv - 110.48 MB
uci_test_reproducido.csv - 28.34 MB


In [21]:
# ============================================================
# 21. VERIFICACIÓN DE ARTEFACTOS EXPORTADOS
# ============================================================

archivos_salida = [
    smart_train_path,
    smart_test_path,
    uci_train_path,
    uci_test_path
]

print("========== ARTEFACTOS GENERADOS ==========\n")

for archivo in archivos_salida:
    estado = "OK" if archivo.exists() else "ERROR"

    print(
        f"{estado:5} | "
        f"{archivo.name:35} | "
        f"{archivo.stat().st_size / 1024**2:.2f} MB"
        if archivo.exists()
        else f"{estado:5} | {archivo.name}"
    )

========== ARTEFACTOS GENERADOS ==========

OK    | smart_home_train_reproducido.csv    | 10.71 MB
OK    | smart_home_test_reproducido.csv     | 2.68 MB
OK    | uci_train_reproducido.csv           | 110.48 MB
OK    | uci_test_reproducido.csv            | 28.34 MB


# Resultado CD2

El proceso de Feature Engineering genera dos conjuntos preparados para modelado.

## Smart Home

- 100.000 observaciones procesadas.
- 20 variables predictoras.
- Variable objetivo: `Energy Consumption (kWh)`.
- Codificación One-Hot de `Appliance Type`.
- Variables temporales cíclicas.
- Indicadores `is_weekend` e `is_peak_hour`.
- Partición aleatoria reproducible 80/20 con `random_state=42`.

Resultados:

- Train: 80.000 registros.
- Test: 20.000 registros.

## UCI Household Power Consumption

- 1.044.506 observaciones válidas después de la preparación.
- 13 variables predictoras.
- Variable objetivo: `Global_active_power`.
- Variables temporales cíclicas.
- Indicadores `is_weekend` e `is_peak_hour`.
- Variables energéticas auxiliares para auditoría.
- Partición cronológica 80/20.

Resultados:

- Train: 835.604 registros.
- Test: 208.902 registros.

La partición temporal de UCI evita utilizar observaciones futuras durante el entrenamiento.

Los datasets resultantes constituyen los artefactos preparados por CD2 para la etapa de modelado CD3.

In [24]:
from pathlib import Path
import shutil

ORIGEN_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/feature_engineering.ipynb"
)

DESTINO_NOTEBOOK = Path(
    "/content/team30_repo/analisis_datos/notebooks/cd2/feature_engineering.ipynb"
)

# Verificación previa
assert ORIGEN_NOTEBOOK.exists(), (
    f"No se encontró el notebook: {ORIGEN_NOTEBOOK}"
)

# Copiar
shutil.copy2(
    ORIGEN_NOTEBOOK,
    DESTINO_NOTEBOOK
)

print("Notebook copiado correctamente.")
print("\nOrigen:")
print(ORIGEN_NOTEBOOK)

print("\nDestino:")
print(DESTINO_NOTEBOOK)

print("\nExiste en repositorio:")
print(DESTINO_NOTEBOOK.exists())

print(
    "\nTamaño:",
    round(DESTINO_NOTEBOOK.stat().st_size / 1024**2, 2),
    "MB"
)

Notebook copiado correctamente.

Origen:
/content/drive/MyDrive/Colab Notebooks/feature_engineering.ipynb

Destino:
/content/team30_repo/analisis_datos/notebooks/cd2/feature_engineering.ipynb

Existe en repositorio:
True

Tamaño: 0.07 MB


In [25]:
%cd /content/team30_repo

!git status --short

print("\n=== CONTENIDO CD2 ===")
!find analisis_datos/notebooks/cd2 -maxdepth 1 -type f | sort

/content/team30_repo
?? analisis_datos/notebooks/cd2/data/smart_home_test_reproducido.csv
?? analisis_datos/notebooks/cd2/data/smart_home_train_reproducido.csv
?? analisis_datos/notebooks/cd2/data/uci_test_reproducido.csv
?? analisis_datos/notebooks/cd2/data/uci_train_reproducido.csv
?? analisis_datos/notebooks/cd2/feature_engineering.ipynb

=== CONTENIDO CD2 ===
analisis_datos/notebooks/cd2/feature_engineering.ipynb
analisis_datos/notebooks/cd2/.gitkeep
analisis_datos/notebooks/cd2/README_CD3.md


In [26]:
from pathlib import Path

DATA_CD2 = Path(
    "/content/team30_repo/analisis_datos/notebooks/cd2/data"
)

archivos_prueba = [
    "smart_home_train_reproducido.csv",
    "smart_home_test_reproducido.csv",
    "uci_train_reproducido.csv",
    "uci_test_reproducido.csv"
]

print("=== LIMPIEZA DE ARCHIVOS DE VALIDACIÓN ===\n")

for nombre in archivos_prueba:
    archivo = DATA_CD2 / nombre

    if archivo.exists():
        archivo.unlink()
        print("ELIMINADO |", nombre)
    else:
        print("NO EXISTE |", nombre)

print("\nLimpieza terminada.")

=== LIMPIEZA DE ARCHIVOS DE VALIDACIÓN ===

ELIMINADO | smart_home_train_reproducido.csv
ELIMINADO | smart_home_test_reproducido.csv
ELIMINADO | uci_train_reproducido.csv
ELIMINADO | uci_test_reproducido.csv

Limpieza terminada.


In [27]:
%cd /content/team30_repo

!git status --short

print("\n=== DATASETS OFICIALES CD2 ===")
!find analisis_datos/notebooks/cd2/data -maxdepth 1 -type f | sort

print("\n=== NOTEBOOK CD2 ===")
!ls -lh analisis_datos/notebooks/cd2/feature_engineering.ipynb

/content/team30_repo
?? analisis_datos/notebooks/cd2/feature_engineering.ipynb

=== DATASETS OFICIALES CD2 ===
analisis_datos/notebooks/cd2/data/smart_home_test.csv
analisis_datos/notebooks/cd2/data/smart_home_train.csv
analisis_datos/notebooks/cd2/data/uci_test.csv
analisis_datos/notebooks/cd2/data/uci_train.csv

=== NOTEBOOK CD2 ===
-rw------- 1 root root 69K Jul 24 21:46 analisis_datos/notebooks/cd2/feature_engineering.ipynb


In [28]:
from pathlib import Path
import json

REPO = Path("/content/team30_repo")

README = (
    REPO
    / "analisis_datos"
    / "notebooks"
    / "cd2"
    / "README_CD3.md"
)

MANIFEST = (
    REPO
    / "analisis_datos"
    / "reports"
    / "cd2"
    / "manifest_cd3.json"
)

print("=" * 70)
print("README_CD3.md ACTUAL")
print("=" * 70)

print(README.read_text(encoding="utf-8"))

print("\n" + "=" * 70)
print("manifest_cd3.json ACTUAL")
print("=" * 70)

with open(MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
))

README_CD3.md ACTUAL

# ENTREGA CD2 -> CD3

## Sistema Inteligente para el Análisis de Consumo Energético

Este directorio contiene los artefactos preparados por el
Científico de Datos 2 para las etapas posteriores del proyecto.

## DATA

smart_home_train.csv
- Dataset de entrenamiento Smart Home.

smart_home_test.csv
- Dataset de prueba Smart Home.

uci_train.csv
- Dataset de entrenamiento UCI.
- Conserva orden cronológico.

uci_test.csv
- Dataset de prueba UCI.
- Comienza exactamente después del último registro TRAIN.

## REPORTS

reporte_entrega_cd2_cd3.txt
- Documentación metodológica completa del procesamiento realizado.

manifest_cd3.json
- Especificación estructurada de datasets, objetivos, predictores
  y estrategia de partición.

## IMPORTANTE PARA CD3

No volver a realizar la división TRAIN/TEST.

No utilizar variables excluidas como predictores.

Para Smart Home:

Target:
Energy Consumption (kWh)

Para UCI:

Target:
Global_active_power

La partición UCI debe conservarse obli